# 0. Problem
## 1581. Customer Who Visited but Did Not Make Any Transactions — Easy
Count, per customer, how many visits had no matching transaction.

Official: https://leetcode.com/problems/customer-who-visited-but-did-not-make-any-transactions/

# 1. Setup

In [ ]:
import pandas as pd
visits_rows=[(1,23),(2,9),(4,30),(5,54),(6,96),(7,54),(8,54)]
transactions_rows=[(2,5,310),(3,5,300),(9,5,200),(12,1,910),(13,2,970)]
visits_pd=pd.DataFrame(visits_rows,columns=["visit_id","customer_id"])
transactions_pd=pd.DataFrame(transactions_rows,columns=["transaction_id","visit_id","amount"])
visits_pd, transactions_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
visits_spark=spark.createDataFrame(visits_rows,["visit_id","customer_id"])
transactions_spark=spark.createDataFrame(transactions_rows,["transaction_id","visit_id","amount"])
visits_spark.createOrReplaceTempView("Visits")
transactions_spark.createOrReplaceTempView("Transactions")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
SELECT v.customer_id, COUNT(*) AS count_no_trans
FROM Visits v
LEFT JOIN Transactions t
  ON v.visit_id=t.visit_id
WHERE t.transaction_id IS NULL
GROUP BY v.customer_id
ORDER BY v.customer_id
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
no_transaction_visits=visits_pd.loc[~visits_pd["visit_id"].isin(transactions_pd["visit_id"])]
result_pd=(no_transaction_visits.groupby("customer_id",as_index=False).agg(count_no_trans=("visit_id","count")).sort_values("customer_id").reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
no_transaction_visits=visits_spark.join(transactions_spark.select("visit_id").distinct(),on="visit_id",how="left_anti")
result_spark=(no_transaction_visits.groupBy("customer_id").agg(F.count("visit_id").alias("count_no_trans")).orderBy("customer_id"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| anti-join | `LEFT JOIN ... IS NULL` | `~.isin(...)` | `left_anti` |
| count/group | `COUNT + GROUP BY` | `.groupby().agg()` | `.groupBy().agg()` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Visits, Transactions

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: visits_pd, transactions_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: visits_spark, transactions_spark